In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers librosa soundfile accelerate

In [ ]:
from google.colab import drive;

In [ ]:
drive.mount('/content/drive')


In [ ]:
!ls "/content/drive/MyDrive/ContextAI"

In [ ]:
!ls "/content/drive/MyDrive"

In [ ]:
!ls "/content/drive/MyDrive/Context AI"

In [ ]:
!ls "/content/drive/MyDrive/Context AI/processed"

In [ ]:
import os, re, json, time
import torch
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

BASE = "/content/drive/MyDrive/Context AI/processed"
MANIFEST_PATH = f"{BASE}/manifest.jsonl"
SEGMENTS_DIR = f"{BASE}/segments"
TRANSCRIPT_LOG = f"{BASE}/transcription.log"

HF_ASR_MODEL_ID = "openai/whisper-large-v3"
TARGET_SR = 16000

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def load_manifest():
    records = []
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def save_manifest(records):
    tmp = MANIFEST_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, MANIFEST_PATH)

records = load_manifest()
total = len(records)
print(f"Total segments to transcribe (ALL, fresh): {total}")

print("Loading model (float32, may take a few minutes)...")
processor = WhisperProcessor.from_pretrained(HF_ASR_MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(HF_ASR_MODEL_ID, torch_dtype=torch.float32)
model = model.to(device)
print("Model loaded on", device)

def transcribe_one(wav_path):
    y, _ = librosa.load(wav_path, sr=TARGET_SR, mono=True)
    inputs = processor(y, sampling_rate=TARGET_SR, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            language="urdu",
            task="transcribe",
            num_beams=3,
            max_new_tokens=225,
        )
    text = processor.batch_decode(generated, skip_special_tokens=True)[0]
    return text.strip()

done = 0
n_errors = 0
start = time.time()

with open(TRANSCRIPT_LOG, "a", encoding="utf-8") as lf:
    lf.write(f"[Colab GPU run - FULL 903, large-v3, fresh restart] 0/{total} segments\n")

for r in records:
    seg_id = r.get("segment_id")
    fname = r.get("filename")
    source_type = r.get("source_type", "")
    wav = f"{SEGMENTS_DIR}/{source_type}/{fname}"
    if not os.path.exists(wav):
        wav = f"{SEGMENTS_DIR}/{fname}"
    if not os.path.exists(wav):
        print(f"MISSING: {fname}")
        n_errors += 1
        done += 1
        continue
    try:
        text = transcribe_one(wav)
        r["transcript"] = text
        r["transcript_model"] = HF_ASR_MODEL_ID
        r["verified"] = False   # reset — sab dobara fresh check hone chahiye
    except Exception as e:
        print(f"ERROR on {seg_id}: {e}")
        n_errors += 1

    done += 1
    if done % 10 == 0 or done == total:
        save_manifest(records)
        elapsed = time.time() - start
        rate = done / elapsed if elapsed else 0
        eta = (total - done) / rate / 60 if rate else 0
        msg = f"{done}/{total} done ({100*done/total:.1f}%) | {rate:.2f} seg/s | ETA {eta:.1f} min"
        print(msg)
        with open(TRANSCRIPT_LOG, "a", encoding="utf-8") as lf:
            lf.write(msg + "\n")

save_manifest(records)
print(f"DONE. total={total} errors={n_errors}")